# Задача 1

## Про FWL-Теорему


In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
 
np.random.seed(42069)

# Пусть у нас есть набор данных, где есть линейная зависимость y от X1 и X2
# При этом X1 и X2 тоже малясь зависимые
df = pd.DataFrame({'x1': np.random.uniform(0, 10, size=1000)})
df['x2'] = 4.9 + df['x1'] * 0.983 + 2.104 * np.random.normal(0, 1.35, size=1000)
df['y'] = 8.643 - 2.34 * df['x1'] + 3.35 * df['x2'] + np.random.normal(0, 1.65, size=1000)
df['const'] = 1
 
# Построим линейную регрессию-МНК из 1, X1 и X2
model = sm.OLS(
    endog=df['y'],
    exog=df[['const', 'x1', 'x2']]
).fit()

# Внимательно смотрим на коэффициент при X2
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.972
Model:                            OLS   Adj. R-squared:                  0.972
Method:                 Least Squares   F-statistic:                 1.754e+04
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        21:09:36   Log-Likelihood:                -1934.3
No. Observations:                1000   AIC:                             3875.
Df Residuals:                     997   BIC:                             3889.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.6842      0.139     62.606      0.000       8.412       8.956
x1            -2.3455      0.027    -88.274      0.000      -2.398      -2.293
x2             3.3544      0.019    178.202      0.000       3.317       3.391
==============================================================================
Omnibus:                        0.497   Durbin-Watson:                   2.031
Prob(Omnibus):                  0.780   Jarque-Bera (JB):                0.394
Skew:                          -0.036   Prob(JB):                        0.821
Kurtosis:                       3.066   Cond. No.                         31.5
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [9]:
# Научим регрессию X1 на X2
model_x2 = sm.OLS(
    endog=df['x2'],
    exog=df[['const', 'x1']]
).fit()

# Научим регрессию X1 на y
model_yx1 = sm.OLS(
    endog=df['y'],
    exog=df[['const', 'x1']]
).fit()

In [11]:
# Полученными регрессиями "предскажем X2 и Y"

df['yx1'] = model_yx1.predict(df[['const', 'x1']])
df['x2x1'] = model_x2.predict(df[['const', 'x1']])

In [12]:
# А затем "отпилим" предсказание из данных
df['y_detrended'] = df['y'] - df['yx1']
df['x2_detrended'] = df['x2'] - df['x2x1']

In [13]:
# Учим модель на "очищенных" переменных и, о Боже, коэффициент при X2 остается "как был"
model_detrended = sm.OLS(
    endog=df['y_detrended'],
    exog=df[['const', 'x2_detrended']]
).fit()
model_detrended.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:            y_detrended   R-squared:                       0.970
Model:                            OLS   Adj. R-squared:                  0.970
Method:                 Least Squares   F-statistic:                 3.179e+04
Date:                Thu, 09 Oct 2025   Prob (F-statistic):               0.00
Time:                        21:12:26   Log-Likelihood:                -1934.3
No. Observations:                1000   AIC:                             3873.
Df Residuals:                     998   BIC:                             3882.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         1.672e-15      0.053   3.16e-14      1.000      -0.104       0.104
x2_detrended     3.3544      0.019    178.292      0.000       3.318       3.391
==============================================================================
Omnibus:                        0.497   Durbin-Watson:                   2.031
Prob(Omnibus):                  0.780   Jarque-Bera (JB):                0.394
Skew:                          -0.036   Prob(JB):                        0.821
Kurtosis:                       3.066   Cond. No.                         2.82
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

Вроде коэффициент тот же. А теперь задача:

Возьмите данные c kaggle, например [отсюда](https://www.kaggle.com/code/malakalaabiad/house-prices-techniques/input) и удостоверьтесь, что FWL-теорема работает для случая нескольких переменных.

# Решение

Оценим влияние признака GrLivArea на SalePrice

In [8]:
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv('data/Train Data.csv')

# Оставляем только необходимые столбцы
df_selected = df[['SalePrice', 'YearBuilt', 'MSSubClass', 'SaleCondition', 'GrLivArea', 'MSZoning']].copy()

# Преобразуем категориальные переменные в dummy-переменные
df_processed = pd.get_dummies(df_selected, columns=['SaleCondition', 'MSZoning'], drop_first=True)

# Преобразуем все булевы столбцы в int (0/1)
bool_columns = df_processed.select_dtypes(include=['bool']).columns
df_processed[bool_columns] = df_processed[bool_columns].astype(int)

# Разделяем на endog и exog
endog = df_processed['SalePrice']
exog = df_processed.drop('SalePrice', axis=1)
exog = sm.add_constant(exog)

model = sm.OLS(
    endog=endog,
    exog=exog
).fit()

# Внимательно смотрим на коэффициент при X2
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:              SalePrice   R-squared:                       0.691
Model:                            OLS   Adj. R-squared:                  0.688
Method:                 Least Squares   F-statistic:                     269.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):               0.00
Time:                        15:50:12   Log-Likelihood:                -17687.
No. Observations:                1460   AIC:                         3.540e+04
Df Residuals:                    1447   BIC:                         3.547e+04
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
=========================================================================================
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                   -1.8e+06   9.03e+04    -19.932      0.000   -1.98e+06   -1.62e+06
YearBuilt               920.5253     46.640     19.737      0.000     829.035    1012.016
MSSubClass             -240.1906     29.971     -8.014      0.000    -298.982    -181.399
GrLivArea                94.5716      2.301     41.106      0.000      90.059      99.085
SaleCondition_AdjLand  1657.2909   2.27e+04      0.073      0.942   -4.29e+04    4.62e+04
SaleCondition_Alloca  -9413.8849   1.36e+04     -0.693      0.489   -3.61e+04    1.72e+04
SaleCondition_Family  -5905.2546   1.09e+04     -0.542      0.588   -2.73e+04    1.55e+04
SaleCondition_Normal    1.23e+04   4677.660      2.629      0.009    3123.633    2.15e+04
SaleCondition_Partial  4.589e+04   6331.586      7.247      0.000    3.35e+04    5.83e+04
MSZoning_FV            1.858e+04   1.56e+04      1.187      0.235   -1.21e+04    4.93e+04
MSZoning_RH            8378.3795    1.8e+04      0.466      0.642   -2.69e+04    4.37e+04
MSZoning_RL             2.35e+04   1.45e+04      1.624      0.105   -4883.408    5.19e+04
MSZoning_RM            2.147e+04   1.46e+04      1.473      0.141   -7118.439    5.01e+04
==============================================================================
Omnibus:                      354.951   Durbin-Watson:                   1.985
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            18206.097
Skew:                           0.142   Prob(JB):                         0.00
Kurtosis:                      20.297   Cond. No.                     1.95e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.95e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [9]:
print(f"Коэффициент GrLivArea в полной модели: {model.params['GrLivArea']:.2f}")

Коэффициент GrLivArea в полной модели: 94.57


In [10]:
# Шаг 2.1: "Очищаем" GrLivArea от влияния всех других переменных
# Создаем матрицу всех переменных кроме GrLivArea
exog_other = exog.drop('GrLivArea', axis=1)

# Регрессия: GrLivArea ~ все другие переменные
model_clean_x = sm.OLS(
    endog=df_processed['GrLivArea'],
    exog=exog_other
).fit()

# Получаем остатки - "очищенный" GrLivArea
grlivarea_residuals = model_clean_x.resid

In [11]:
# Шаг 2.2: "Очищаем" SalePrice от влияния всех других переменных
# Регрессия: SalePrice ~ все другие переменные (кроме GrLivArea)
model_clean_y = sm.OLS(
    endog=endog,
    exog=exog_other
).fit()

# Получаем остатки - "очищенный" SalePrice
saleprice_residuals = model_clean_y.resid

In [21]:
# Шаг 2.3: Регрессия остатков на остатки
model_fwl = sm.OLS(
    endog=saleprice_residuals,
    exog=grlivarea_residuals
).fit()

# Не добавляем константу, так как остатки уже центрированы
fwl_coefficient = model_fwl.params
fwl_coefficient

x1    94.571609
dtype: float64

Как видим, коэффиценты равны, что и требовалось показать

# Задача 2

Возьмите [данные](https://www.kaggle.com/datasets/mkechinov/ecommerce-events-history-in-cosmetics-shop) с Kaggle и оцените равномерность разбиения их на группы (для будущего АБ-теста) с помощью различных видов хеширования:

1. md5
2. sha256
3. Улучшится ли равномерность, если вместо одинарного использования md5 применить [вот такую](https://towardsdatascience.com/assign-experiment-variants-at-scale-in-a-b-tests-e80fedb2779d) двухуровневую процедуру с тем же md5, проверить на тех же данных

In [34]:
import pandas as pd

df = pd.read_csv('data/2019-Dec.csv')

users = df['user_id'].dropna().unique()
print(f"Количество уникальных пользователей для тестирования: {len(users)}")

Количество уникальных пользователей для тестирования: 370154


In [141]:
import hashlib

def simple_hash(
        user_id: int,
        hash_function: str ='md5',
        num_buckets: int =100,
) -> int:
    user_str = str(user_id)

    if hash_function == 'md5':
        hash_obj = hashlib.md5(user_str.encode())
    elif hash_function == 'sha256':
        hash_obj = hashlib.sha256(user_str.encode())
    else:
        raise ValueError("Unsupported hash function")

    hash_hex = hash_obj.hexdigest()[:8]
    hash_int = int(hash_hex, 16)

    bucket = hash_int % num_buckets
    return bucket


def two_level_hash(
    user_id: int,
    num_buckets: int = 100,
) -> int:
    EXPOSURE_RATE = 1.0
    EXPERIMENT_SALT = "analytics_hw_2"

    # Шаг 1: Проверка участия в эксперименте
    exposure_string = f"{EXPERIMENT_SALT}{user_id}Exposure"
    exposure_hash = int(hashlib.md5(exposure_string.encode()).hexdigest(), 16)
    exposure_random = exposure_hash % 100

    if exposure_random >= EXPOSURE_RATE * 100:
        return -1  # Пользователь исключен

    # Шаг 2: Назначение бакета
    bucket_string = f"{EXPERIMENT_SALT}{user_id}Bucket"
    bucket_hash = int(hashlib.md5(bucket_string.encode()).hexdigest(), 16)
    bucket_random = bucket_hash % num_buckets

    return bucket_random


In [166]:
from scipy import stats

def calculate_ab_test_metrics(bucket_assignments, num_buckets, method_name):
    # Подсчет количества наблюдений в каждом бакете
    bucket_counts = pd.Series(bucket_assignments).value_counts().sort_index()
    bucket_counts = bucket_counts.reindex(range(num_buckets), fill_value=0)

    total_users = len(bucket_assignments)
    expected_per_bucket = total_users / num_buckets

    # Основные статистические тесты
    chi2_stat, chi2_pvalue = stats.chisquare(bucket_counts)

    # Практические метрики для A/B тестирования
    cv = bucket_counts.std() / bucket_counts.mean()  # Коэффициент вариации
    max_deviation = abs(bucket_counts - expected_per_bucket).max()
    max_deviation_percent = (max_deviation / expected_per_bucket) * 100

    # Оценка неравномерности
    imbalance_ratio = bucket_counts.max() / bucket_counts.min() if bucket_counts.min() > 0 else float('inf')

    metrics = {
        'total_users': total_users,
        'chi2_pvalue': chi2_pvalue,
        'coefficient_of_variation': cv,
        'max_deviation_percent': max_deviation_percent,
        'imbalance_ratio': imbalance_ratio,
        'is_uniform_05': chi2_pvalue > 0.05,  # Равномерность на уровне 5%
        'is_uniform_01': chi2_pvalue > 0.01,  # Равномерность на уровне 1%
    }

    return metrics


In [167]:
def test_hashing_uniformity(users, num_buckets=100, sample_size=100000):
    if len(users) > sample_size:
        test_users = np.random.choice(users, size=sample_size, replace=False)
    else:
        test_users = users

    print(f"Тестируем на выборке из {len(test_users)} пользователей...")
    print(f"Количество бакетов: {num_buckets}")
    print(f"Ожидаемое количество пользователей на бакет: {len(test_users) / num_buckets:.1f}")

    methods = {
        'md5': lambda uid: simple_hash(uid, 'md5', num_buckets),
        'sha256': lambda uid: simple_hash(uid, 'sha256', num_buckets),
        'two_step_md5': lambda uid: two_level_hash(uid, num_buckets)
    }

    metrics_by_method = {}

    for method_name, hash_func in methods.items():
        # Распределяем пользователей по бакетам
        buckets = [hash_func(uid) for uid in test_users]

        # Рассчитываем метрики
        metrics = calculate_ab_test_metrics(buckets, num_buckets, method_name)
        metrics_by_method[method_name] = metrics  # Добавляем в мапу
        print(f"{method_name}:")
        print(f"   ✓ p-value: {metrics['chi2_pvalue']:.6f}")
        # насколько группы отличаются по размеру
        print(f"   ✓ Коэффициент вариации: {metrics['coefficient_of_variation']:.4f}")
        print(f"   ✓ Макс. отклонение: {metrics['max_deviation_percent']:.2f}%")

    return metrics_by_method

In [178]:
results = test_hashing_uniformity(users, num_buckets=100, sample_size=len(users))

Тестируем на выборке из 370154 пользователей...
Количество бакетов: 100
Ожидаемое количество пользователей на бакет: 3701.5
md5:
   ✓ p-value: 0.527558
   ✓ Коэффициент вариации: 0.0163
   ✓ Макс. отклонение: 5.25%
sha256:
   ✓ p-value: 0.870435
   ✓ Коэффициент вариации: 0.0151
   ✓ Макс. отклонение: 4.96%
two_step_md5:
   ✓ p-value: 0.900402
   ✓ Коэффициент вариации: 0.0149
   ✓ Макс. отклонение: 4.42%


Все значение p-value > 0.05, то есть распределение равномерное.

Максимально отклонение варьируется в районе 5%, что тоже свидетельствует о равномерности распределения.

Коэффициент вариации - показывает, насколько группы отличаются по размеру, все значения в рамках 0.01, что указывает на равномерность классов.

Eсли вместо одинарного использования md5 применить двухуровневую процедуру, то равномерность улучшится

# Задача 3

Про эквивалентность или не эквивалентность разных методов подсчета квантилей

Сгенерируйте 2 выборки длины, например, 10000 из:

1. Нормального
2. Логнормального
3. Экспоненциального

Распределений с наперед заданными параметрами, так чтобы вы могли однозначно посчитать разницу медиан (используя теорвер и википедию)



Проверьте, какой по этим выборкам будет получаться 95% доверительный интервал на разницу медиан, если его посчитать с помощью:

1. Пуассоновского бутстрепа
2. [Подгонки](https://engineering.atspotify.com/2022/03/comparing-quantiles-at-scale-in-online-a-b-testing/) от Spotify
3. [Подгонки](https://www.evanmiller.org/bootstrapping-sample-medians.html) результатов бутстрепа от Эвана Миллера (это то самое, с Бесселями)
4. [Метода Прайса-Боннетта](https://www.tandfonline.com/doi/abs/10.1080/00949650212140)

Что вы можете сказать о работоспособности методов?

(можно попробовать подать на вход какие-то другие распределения, как бы провести "стресс-тест" метода)

## 1. Пуассоновский бутстреп

In [2]:
def poisson_bootstrap_weights(n, n_bootstrap=1000, lam=1):
    # Размерность (n_bootstrap, n)
    return np.random.poisson(lam, (n_bootstrap, n))

def poisson_bootstrap_statistic(data, statistic_func, n_bootstrap=1000, lam=1):
    n = len(data)
    weights = poisson_bootstrap_weights(n, n_bootstrap, lam)

    # Векторизация через broadcasting и masked array
    # Умножаем данные на веса и считаем статистику для каждой строки
    bootstrap_stats = np.array([
        statistic_func(np.repeat(data, w)) for w in weights
    ])

    return bootstrap_stats

def poisson_bootstrap_diff_median_ci(sample1, sample2, n_bootstrap=1000, alpha=0.05):
    boot_median1 = poisson_bootstrap_statistic(sample1, np.median, n_bootstrap=n_bootstrap)
    boot_median2 = poisson_bootstrap_statistic(sample2, np.median, n_bootstrap=n_bootstrap)
    return np.quantile(boot_median1 - boot_median2, [alpha/2, 1-alpha/2])

## 2. Schultzberg and Ankargren (Spotify) method

In [3]:
from numpy.random import binomial

def spotify_diff_median_ci(sample1, sample2, n_bootstrap=1000, alpha=0.05):
    quantile=0.5
    sample1_sorted = np.sort(sample1)
    sample2_sorted = np.sort(sample2)
    indices = binomial(n + 1, quantile, n_bootstrap)

    diff_medians = sample1_sorted[indices] - sample2_sorted[indices]
    return np.quantile(diff_medians, [alpha/2, 1-alpha/2])

## 3. Evan Miller Method

In [4]:
from scipy.stats import poisson
from math import sqrt, exp
from scipy.special import i0e, factorial, jve

def exact_bessel_dist_prob(i, n):
    arg = 2 * sqrt((n - i) * (i - 1))
    total = 0
    if arg == 0:
        poisson_dist = poisson(n - 1)
        for k in range(1, 21):
            delta = (0.5 * poisson_dist.pmf(k) + poisson_dist.cdf(k - 1)) / factorial(k)
            total += delta
        return exp(-1) * total
    term = 1
    for eta in range(1, 21):
        prefix = ((n - i) / (i - 1)) ** (eta / 2) + ((i - 1) / (n - i)) ** (eta / 2)
        term += 1 / factorial(eta)
        delta = (0.5 / factorial(eta) + exp(1) - term) * prefix * jve(eta, arg)
        total += delta
    return exp(arg - n) * ((exp(1) - 1) * i0e(arg) + total)


def approx_bessel_dist_prob(i, n):
    arg = 2 * sqrt((n + 1 - i) * i)
    return 2 * exp(arg - n - 1) * i0e(arg)

class PoissonBootstrapSampleMedianDistribution:
    def __init__(self, n, approx=True):
        self.n = n
        if approx:
            probs = np.array([approx_bessel_dist_prob(i, n) for i in range(1, n+1)])
        else:
            probs = np.array([exact_bessel_dist_prob(i, n) for i in range(1, n+1)])
        # Нормируем
        probs /= probs.sum()
        self.cdf = np.cumsum(probs)

    def sample_indices(self, n_samples):
        """Генерируем индексы по кумулятивной функции"""
        u = np.random.rand(n_samples)

        return np.searchsorted(self.cdf, u)

def evan_miller_diff_median_ci(sample1, sample2, n_bootstrap=1000, alpha=0.05, approx=True):
    sample1_sorted = np.sort(sample1)
    sample2_sorted = np.sort(sample2)

    pb_dist = PoissonBootstrapSampleMedianDistribution(n, approx=approx)
    inidices = pb_dist.sample_indices(n_bootstrap)
    return np.quantile(sample1_sorted[inidices] - sample2_sorted[inidices], [alpha/2, 1-alpha/2])

## 4. Метод Прайса-Боннетта

$$CI_{diff}=(Z^1−Z^2)±z_{α/2}Var(Z^1)+Var(Z^2)$$

$$Var(Z_j)=(\frac{(Y_{(n_j−c_j+1)j}−Y_{(c_j)j})}{2z_j})^2$$

> The proposed methods are easy to
compute and perform well in small samples.

In [5]:
import numpy as np
from scipy.stats import norm, binom

def bonett_price_ci(sample1, sample2, alpha=0.05):
    """
    Реализация метода Bonett & Price (2002)
    для доверительных интервалов разности медиан.
    """
    y1, y2 = np.sort(sample1), np.sort(sample2)
    med1, med2 = np.median(y1), np.median(y2)

    def var_median(y):
        n = len(y)
        c = int(round((n + 1) / 2 - np.sqrt(n)))
        c = max(1, c)  # защита от c=0
        # p_j = сумма биномиальных вероятностей
        pj = binom.cdf(c - 1, n, 0.5)
        # z_j = F^-1(1 - p_j / 2)
        zj = norm.ppf(1 - pj / 2)
        return ((y[n - c] - y[c - 1]) / (2 * zj)) ** 2

    var1 = var_median(y1)
    var2 = var_median(y2)

    z_crit = norm.ppf(1 - alpha / 2)
    diff = med1 - med2
    se_diff = np.sqrt(var1 + var2)
    ci_diff = np.array([diff - z_crit * se_diff, diff + z_crit * se_diff])

    return ci_diff

In [6]:
def report_of_diff_median_by_samples(sample1, sample2, alpha=0.05, n_bootstrap=1000):
    poisson_bootstrap_ci =  poisson_bootstrap_diff_median_ci(sample1, sample2, n_bootstrap, alpha)
    spotify_ci = spotify_diff_median_ci(sample1, sample2, n_bootstrap, alpha)
    evan_miller_ci_approx = evan_miller_diff_median_ci(sample1, sample2, n_bootstrap, alpha, approx=True)
    evan_miller_ci_exact = evan_miller_diff_median_ci(sample1, sample2, n_bootstrap, alpha, approx=False)
    bonett_ci = bonett_price_ci(sample1, sample2, alpha)
    return str(
        f"  poisson_bootstrap_ci:{poisson_bootstrap_ci}\n"
        f"  spotify_ci:{spotify_ci}\n"
        f"  evan_miller_ci_approx:{evan_miller_ci_approx}\n"
        f"  evan_miller_ci_exact:{evan_miller_ci_exact}\n"
        f"  bonett_price_ci:{bonett_ci}\n")

## Нормальное распредление

In [8]:
np.random.seed(42069)
n = 10000
n_bootstrap = 1000
alpha = 0.05

mu1, sigma1 = 10, 2
sample_normal1 = np.random.normal(mu1, sigma1, n)

# Теоретическая медиана для нормального распределения = μ
theoretical_median_normal1 = mu1

mu2, sigma2 = 5, 8
sample_normal2 = np.random.normal(mu2, sigma2, n)

theoretical_median_normal2 = mu2

theta_theoretical = mu1 - mu2

In [489]:
report = report_of_diff_median_by_samples(sample_normal1, sample_normal2, alpha=0.05, n_bootstrap=1000)
print(f"Diff medians of normal distribution\n"
      f"  theoretical:{theta_theoretical}\n"
      f"{report}")

Diff medians of distribution
  theoretical:5
  poisson_bootstrap_ci:[4.84387691 5.21863549]
  spotify_ci:[4.89444973 5.17971793]
  evan_miller_ci_approx:[4.8929845  5.17545377]
  evan_miller_ci_exact:[4.94609687 5.13802766]
  bonett_price_ci:[4.85081303 5.18912213]



In [13]:
n = 10000
# Параметры для логнормального: μ и σ - параметры соответствующего нормального распределения
mu_ln1, sigma_ln1 = 9, 1
sample_lognormal1 = np.random.lognormal(mu_ln1, sigma_ln1, n)

# Теоретическая медиана для логнормального: exp(μ)
theoretical_median_ln1 = np.exp(mu_ln1)

mu_ln2, sigma_ln2 = 5, 8
sample_lognormal2 = np.random.lognormal(mu_ln1, sigma_ln2, n)

theoretical_median_ln2 = np.exp(mu_ln2)

theta_theoretical = theoretical_median_ln1 - theoretical_median_ln2
theta_theoretical

np.float64(7954.670768472808)

In [14]:
report = report_of_diff_median_by_samples(sample_lognormal1, sample_lognormal2, alpha=0.05, n_bootstrap=5000)
print(f"Diff medians of lognormal distribution\n"
      f"  theoretical:{theta_theoretical}\n"
      f"{report}")

Diff medians of lognormal distribution
  theoretical:7954.670768472808
  poisson_bootstrap_ci:[-836.54835105 1932.37720775]
  spotify_ci:[-688.31678427 1786.0806271 ]
  evan_miller_ci_approx:[-642.79477014 1795.4269299 ]
  evan_miller_ci_exact:[-102.81632664 1380.43590629]
  bonett_price_ci:[-549.61043732 1946.8454687 ]



Оценки смещённые, неточные, не отображают

## Экспоненциальное распредление

In [531]:
n = 10000
lam1 = 0.01
sample_exp1 = np.random.exponential(1/lam1, n)

# Теоретическая медиана для экспоненциального: ln(2)/λ
theoretical_median_exp1 = np.log(2) / lam1

lam2 = 0.08
sample_exp2 = np.random.exponential(1/lam2, n)

# Теоретическая медиана для экспоненциального: ln(2)/λ
theoretical_median_exp2 = np.log(2) / lam2

theta_theoretical = theoretical_median_exp1 - theoretical_median_exp2
theta_theoretical

np.float64(60.650378298995214)

In [532]:
report = report_of_diff_median_by_samples(sample_exp1, sample_exp2, alpha=0.05, n_bootstrap=1000)
print(f"Diff medians of exponential distribution\n"
      f"  theoretical:{theta_theoretical}\n"
      f"{report}")

Diff medians of exponential distribution
  theoretical:60.650378298995214
  poisson_bootstrap_ci:[58.07360905 61.80894767]
  spotify_ci:[58.27388299 61.58538876]
  evan_miller_ci_approx:[58.36474099 61.49382081]
  evan_miller_ci_exact:[58.75729959 60.98986221]
  bonett_price_ci:[58.07113521 61.334255  ]



# Заключение

Все методы хорошо работают на нормальном и экспоненциальном распределении.

Все методы одинаково плохо приближают доверительный интервал на логнормальном распределением.
Есть несколько гипотез, почему так происходит:
1. Логнормальная модель имеет сильно скошенное распределение. Большая часть данных сконцентрирована слева, а редкие большие наблюдения сильно влияют на порядок статистик и на вариабельность оценок квантилей
2. Бутстрэп пытается аппроксимировать распределение статистики с помощью эмпирической выборки. При сильной скошенности эффективная «полезная» масса наблюдений вокруг истинной медианы меньше, поэтому эмпирический буфер вокруг медианы беден